# 02 — Translation Pipeline Overview

Some initial descriptive statistics on the full pipeline output across all languages and services.
This notebook answers the basic questions before analysis:
- How many languages have translations from each service?
- What is the coverage gap across the 185 ISO 639-1 languages?
- Which services consistently fail on which language families?
- Where does the pipeline produce translations vs. fall silent?

In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df

DATA_DIR = get_data_directory_path()
TERMS = ["Digital Humanities"]
VARIANTS = ["minimal", "expert_persona", "native_rationale", "judge"]

SERVICE_COLS = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
    "Ollama": "ollama_translated_term",
    "OpenAI": "openai_translated_term",
    "Claude": "claude_translated_term",
    "Gemini": "gemini_translated_term",
}

print(f"Data directory: {DATA_DIR}")

Data directory: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets


## 1.1 Load Pipeline Outputs

In [2]:
TARGET_TERMS = ["Digital Humanities"]

def load_all_variants(data_dir, term, variants=VARIANTS):
    """Merge per-service files for all variants into one concatenated DataFrame."""
    dfs = []
    term_slug = term.lower().replace(" ", "_")
    for variant in variants:
        df = load_variant_df(data_dir, term_slug, variant)
        if df is not None:
            df["term_source_query"] = term
            dfs.append(df)
        else:
            print(f"  ⚠ No data for variant: {variant}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

all_dfs = {term: load_all_variants(DATA_DIR, term) for term in TARGET_TERMS}
for term, df in all_dfs.items():
    print(f"{term}: {len(df)} rows across {df['prompt_variant'].nunique()} variants")

all_terms_df = all_dfs[TARGET_TERMS[0]]

Digital Humanities: 3432 rows across 4 variants


## 1.2 Coverage by Service

How many of the 269 languages have a translation from each service?

In [3]:
BASELINE_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
LLM_SERVICES = {
    "Ollama": "ollama_translated_term",
    "OpenAI": "openai_translated_term",
    "Claude": "claude_translated_term",
    "Gemini": "gemini_translated_term",
}

term = TARGET_TERMS[0]
df = all_dfs[term]

ref = df[df["prompt_variant"] == VARIANTS[0]]
total = ref["language_code"].nunique()

baseline_rows = []
for service, col in BASELINE_SERVICES.items():
    n = ref[col].notna().sum() if col in ref.columns else 0
    baseline_rows.append({"service": service, "n": int(n), "total": total})
baseline_cov = pd.DataFrame(baseline_rows)

llm_rows = []
for variant in VARIANTS:
    vdf = df[df["prompt_variant"] == variant]
    v_total = vdf["language_code"].nunique()
    for service, col in LLM_SERVICES.items():
        n = vdf[col].notna().sum() if col in vdf.columns else 0
        llm_rows.append({"service": service, "variant": variant, "n": int(n), "total": v_total})
llm_cov = pd.DataFrame(llm_rows)

# Single named selection — added once to the composed chart, not to sub-charts
selection = alt.selection_point(name="service_sel", fields=["service"], bind="legend")
color_scale = alt.Color("service:N", scale=alt.Scale(scheme="tableau10"), title="Service")
fade = alt.condition(selection, alt.value(1), alt.value(0.2))

# Chart 1: baseline coverage
baseline_bars = alt.Chart(baseline_cov).mark_bar().encode(
    y=alt.Y("service:N", sort="-x", title=None),
    x=alt.X("n:Q", scale=alt.Scale(domain=[0, total]), title="languages translated"),
    color=color_scale,
    opacity=fade,
    tooltip=["service:N", alt.Tooltip("n:Q", title="languages"), alt.Tooltip("total:Q", title="total")],
).properties(width=350, height=160, title=f"Baseline service coverage (n={total} languages, prompt-invariant)")

baseline_text = baseline_bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text="n:Q", color=alt.value("black"), opacity=alt.value(1),
)

# Chart 2: LLM coverage by variant
llm_bars = alt.Chart(llm_cov).mark_bar().encode(
    y=alt.Y("variant:N", sort=VARIANTS, title=None),
    x=alt.X("n:Q", title="languages translated"),
    yOffset="service:N",
    color=color_scale,
    opacity=fade,
    tooltip=["service:N", "variant:N", alt.Tooltip("n:Q", title="languages"), alt.Tooltip("total:Q", title="total")],
).properties(width=380, height=280, title="LLM service coverage by prompt variant")

# Chart 3: connected scatter — service lines across variants
scatter_line = alt.Chart(llm_cov).mark_line(opacity=0.35, strokeWidth=1.5).encode(
    x=alt.X("variant:N", sort=VARIANTS, title=None),
    y=alt.Y("n:Q", title="languages translated", scale=alt.Scale(zero=False)),
    color=color_scale,
    detail="service:N",
)
scatter_pts = alt.Chart(llm_cov).mark_point(filled=True, size=80).encode(
    x=alt.X("variant:N", sort=VARIANTS, title=None),
    y=alt.Y("n:Q", title="languages translated", scale=alt.Scale(zero=False)),
    color=color_scale,
    opacity=fade,
    tooltip=["service:N", "variant:N", alt.Tooltip("n:Q", title="languages")],
)
scatter = (scatter_line + scatter_pts).properties(
    width=380, height=220,
    title="Coverage per service across variants (flat line = variant has no effect)",
)

# Add the selection parameter once to the composed chart
((baseline_bars + baseline_text) & llm_bars & scatter).add_params(selection)

/Users/zleblanc/.virtualenvs/coding-dh-env/lib/python3.13/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.VConcatChart(...)

In [4]:
all_terms_df[all_terms_df.language_code == "sr"].to_clipboard()

## 1.3 Coverage Heatmap by Language Family

In [12]:
HEATMAP_SERVICES = {
    "Wikipedia":        "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT":          "enmt_translated_term",
    "Lingvanex":        "lingvanex_translated_term",
    "OpenAI":           "openai_translated_term",
    "Claude":           "claude_translated_term",
    "Gemini":           "gemini_translated_term",
    "Ollama":           "ollama_translated_term",
}
service_order = list(HEATMAP_SERVICES.keys())

term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
baseline["language_family"] = baseline["language_code"].apply(get_language_family)

other_langs = baseline[baseline["language_family"] == "Other"]["language_code"].tolist()
if other_langs:
    print(f"⚠ Ungrouped languages: {other_langs}")
else:
    print("✓ All languages assigned to a family")

family_counts = baseline["language_family"].value_counts().to_dict()

rows = []
for family, n_fam in sorted(family_counts.items(), key=lambda x: -x[1]):
    fam_df = baseline[baseline["language_family"] == family]
    label = f"{family} ({n_fam})"
    for service, col in HEATMAP_SERVICES.items():
        n_translated = int(fam_df[col].notna().sum()) if col in fam_df.columns else 0
        pct = round(n_translated / n_fam * 100, 1) if n_fam else 0.0
        rows.append({"Family": label, "Service": service, "Coverage": pct, "N": n_translated, "Family_size": n_fam})
coverage_long = pd.DataFrame(rows)

family_order = [f"{f} ({family_counts[f]})" for f in sorted(family_counts.keys(), key=lambda k: -family_counts[k])]
chart = alt.Chart(coverage_long).mark_rect().encode(
    x=alt.X("Service:N", sort=service_order, title="Translation Service"),
    y=alt.Y("Family:N", sort=family_order, title="Language Family"),
    color=alt.Color("Coverage:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, 100]),
        legend=alt.Legend(title="% coverage"),
    ),
    tooltip=[
        "Family", "Service",
        alt.Tooltip("Coverage:Q", format=".1f", title="% coverage"),
        alt.Tooltip("N:Q", title="languages translated"),
        alt.Tooltip("Family_size:Q", title="family size"),
    ],
).properties(
    title=alt.Title(
        "Languages Translated by Service and Language Family",
        subtitle="Baseline services (left) run once; LLM services (right) shown for minimal variant.",
    ),
    width=520, height=420,
)
text = chart.mark_text(baseline="middle", fontSize=8).encode(
    text=alt.Text("Coverage:Q", format=".0f"),
    color=alt.condition(alt.datum.Coverage > 60, alt.value("white"), alt.value("black")),
)
(chart + text)

⚠ Ungrouped languages: ['see also: test languages at the Wikimedia Incubator', nan]


alt.LayerChart(...)

## 1.4 Languages with Zero Coverage

These are the most interesting cases for the paper — languages where the pipeline
produced nothing at all. Are they low-resource languages? Script-diverse languages?
Languages where 'Digital Humanities' genuinely has no circulation?

In [15]:
term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
baseline["language_family"] = baseline["language_code"].apply(get_language_family)

svc_cols = [c for c in HEATMAP_SERVICES.values() if c in baseline.columns]
baseline["n_services"] = baseline[svc_cols].notna().sum(axis=1)

TIERS = {"0 — none": (0,0), "1–2 — sparse": (1,2), "3–5 — partial": (3,5), "6–9 — rich": (6,9)}
def tier(n):
    for label, (lo, hi) in TIERS.items():
        if lo <= n <= hi: return label
    return "other"
baseline["tier"] = baseline["n_services"].apply(tier)

# Histogram: overall coverage depth
tier_order = list(TIERS.keys())
hist = alt.Chart(baseline).mark_bar().encode(
    x=alt.X("n_services:O", title="Services that translated this language", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Number of languages"),
    color=alt.Color("tier:N", sort=tier_order,
        scale=alt.Scale(domain=tier_order, range=["#d32f2f","#ff9800","#1976d2","#388e3c"]),
        title="Coverage tier"),
    tooltip=["n_services:O", "count():Q", "tier:N"],
).properties(width=360, height=240, title="Service coverage depth per language")

# All zero + sparse languages in one chart
low_cov = baseline[baseline["n_services"] <= 2][
    ["language_code","language_name","language_family","n_services","tier"]
].sort_values(["n_services","language_family"]).reset_index(drop=True)

print(f"Zero coverage: {(baseline['n_services']==0).sum()} | Sparse (1-2): {((baseline['n_services']>=1)&(baseline['n_services']<=2)).sum()}")

low_chart = alt.Chart(low_cov).mark_circle(size=90).encode(
    y=alt.Y("language_name:N",
        sort=alt.EncodingSortField("n_services", order="ascending"),
        title=None),
    x=alt.X("n_services:Q", title="services that translated it",
        scale=alt.Scale(domain=[-0.3, 2.3]),
        axis=alt.Axis(values=[0,1,2])),
    color=alt.Color("language_family:N", title="Language family"),
    shape=alt.Shape("tier:N", sort=tier_order,
        scale=alt.Scale(range=["cross","circle","circle"]),
        title="Coverage tier"),
    tooltip=["language_code:N","language_name:N","language_family:N","n_services:Q"],
).properties(
    width=280,
    height=max(120, len(low_cov) * 14),
    title=f"Zero & sparse coverage languages (n={len(low_cov)})",
)

hist 

Zero coverage: 0 | Sparse (1-2): 5


alt.Chart(...)

In [16]:
low_chart

alt.Chart(...)

## 1.5 Error Log Analysis

What do the error logs tell us about *why* translations failed?

In [17]:
SERVICE_NAMES = {
    "claude": "Claude", "enmt": "EasyNMT",
    "gemini": "Gemini", "gt": "Google Translate", "lingvanex": "Lingvanex",
    "ollama": "Ollama", "openai": "OpenAI", "wikipedia": "Wikipedia",
}

error_dir = os.path.join(DATA_DIR, "error_logs")
error_rows = []

if os.path.exists(error_dir):
    for fname in sorted(os.listdir(error_dir)):
        if not fname.endswith(".csv"): continue
        key = fname.replace("_translation_errors.csv", "")
        name = SERVICE_NAMES.get(key, key)
        try:
            edf = read_csv_file(os.path.join(error_dir, fname))
            if "status_code" in edf.columns:
                for code, cnt in edf["status_code"].value_counts().items():
                    error_rows.append({"service": name, "status": str(code), "count": int(cnt)})
            else:
                error_rows.append({"service": name, "status": "unknown", "count": len(edf)})
        except Exception as e:
            print(f"Could not read {fname}: {e}")

error_df = pd.DataFrame(error_rows)

if error_df.empty:
    print("No error rows found — skipping error charts.")
else:
    totals = error_df.groupby("service")["count"].sum().reset_index().sort_values("count", ascending=False)
    svc_order = totals["service"].tolist()

    total_bar = alt.Chart(totals).mark_bar().encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("count:Q", title="total errors logged"),
        color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
        tooltip=["service:N", "count:Q"],
    ).properties(width=320, height=220, title="Total errors per service")

    total_text = total_bar.mark_text(align="left", dx=4, fontSize=9).encode(
        text="count:Q", color=alt.value("black"))

    status_bar = alt.Chart(error_df).mark_bar().encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("count:Q", title="error count"),
        color=alt.Color("status:N", title="HTTP status", scale=alt.Scale(scheme="set2")),
        order=alt.Order("count:Q", sort="descending"),
        tooltip=["service:N", "status:N", "count:Q"],
    ).properties(width=320, height=220, title="Error breakdown by status code")

    display((total_bar + total_text) | status_bar)

alt.HConcatChart(...)

## 1.6 Translation Length Distribution

Do translations cluster in length? Very short translations may be transliterations; very long ones may be definitions rather than term equivalents.

In [18]:
term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()

length_rows = []
for service, col in HEATMAP_SERVICES.items():
    if col not in baseline.columns: continue
    for wc in baseline[col].dropna().astype(str).str.split().str.len():
        length_rows.append({"Service": service, "Word Count": int(wc)})
length_df = pd.DataFrame(length_rows)

bars = alt.Chart().mark_bar(opacity=0.85).encode(
    x=alt.X("Word Count:Q", bin=alt.Bin(extent=[1,12], step=1), title="Word count"),
    y=alt.Y("count():Q", title="N languages"),
    color=alt.Color("Service:N", scale=alt.Scale(scheme="tableau10"), legend=None),
    tooltip=["Service", alt.Tooltip("count():Q", title="Count")],
).properties(width=155, height=110)

median_rule = alt.Chart().mark_rule(color="red", strokeDash=[4,2]).encode(
    x=alt.X("median(Word Count):Q"),
    tooltip=[alt.Tooltip("median(Word Count):Q", format=".1f", title="Median")],
)

alt.layer(bars, median_rule, data=length_df).facet(
    facet=alt.Facet("Service:N", sort=list(HEATMAP_SERVICES.keys()), title=None),
    columns=3,
    title=f"Translation Word Count Distribution — {term} (red line = median)",
).display()

alt.FacetChart(...)

## 1.7 Service Gaps

No service is the *sole* translator for any language — every language was covered by at least 4 services.
The more useful question is: **which services have the most gaps**, and **which services show up
for the hardest-to-translate languages** (those covered by the fewest services overall)?

In [19]:
term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
baseline["language_family"] = baseline["language_code"].apply(get_language_family)

svc_cols = {s: c for s, c in HEATMAP_SERVICES.items() if c in baseline.columns}
total_langs = len(baseline)

# For each language, which services translated it?
coverage_matrix = pd.DataFrame(
    {s: baseline[c].notna().values for s, c in svc_cols.items()},
    index=baseline["language_code"].values,
)
coverage_matrix["n_total"] = coverage_matrix[list(svc_cols.keys())].sum(axis=1)

# Gap count per service: languages it did NOT translate
gap_rows = []
for service in svc_cols:
    present = int(coverage_matrix[service].sum())
    absent = total_langs - present
    # Among the languages with fewest services (n_total == min), is this service present?
    sparse = coverage_matrix[coverage_matrix["n_total"] <= 5]
    present_in_sparse = int(coverage_matrix.loc[sparse.index, service].sum())
    gap_rows.append({
        "service": service,
        "translated": present,
        "missed": absent,
        "sparse_present": present_in_sparse,
        "sparse_total": len(sparse),
    })
gap_df = pd.DataFrame(gap_rows).sort_values("missed", ascending=False)

# Chart 1: missed languages per service
svc_sort = gap_df["service"].tolist()
gap_bar = alt.Chart(gap_df).mark_bar().encode(
    y=alt.Y("service:N", sort=svc_sort, title=None),
    x=alt.X("missed:Q", title=f"languages not translated (of {total_langs})"),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N", "missed:Q", "translated:Q"],
).properties(width=340, height=220, title="Gaps per service — languages not translated")
gap_text = alt.Chart(gap_df).mark_text(align="left", dx=4, fontSize=9).encode(
    y=alt.Y("service:N", sort=svc_sort),
    x="missed:Q",
    text="missed:Q",
)

# Chart 2: which services are present for sparse languages (n_total <= 5)
sparse_bar = alt.Chart(gap_df).mark_bar().encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("sparse_present", order="descending"), title=None),
    x=alt.X("sparse_present:Q",
        scale=alt.Scale(domain=[0, gap_df["sparse_total"].iloc[0]]),
        title=f"languages translated (of {gap_df['sparse_total'].iloc[0]} hardest)"),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N", "sparse_present:Q",
             alt.Tooltip("sparse_total:Q", title="hardest languages total")],
).properties(width=340, height=220, title="Coverage among hardest languages (≤5 services)")
sparse_text = alt.Chart(gap_df).mark_text(align="left", dx=4, fontSize=9).encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("sparse_present", order="descending")),
    x="sparse_present:Q",
    text="sparse_present:Q",
)

(gap_bar + gap_text) | (sparse_bar + sparse_text)

alt.HConcatChart(...)

## 1.8 Script & Directionality

RTL languages and CJK-script languages are structurally harder for translation pipelines.
Does coverage drop for these groups?

In [ ]:
# Build script-group sets from the comprehensive language metadata
_lang_meta = read_csv_file(os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv"))
_lang_meta = _lang_meta.dropna(subset=["language_code"])

# RTL: CLDR-derived directionality column (78 codes); already has FORCE_LTR overrides applied
# — more complete than any hardcoded list and correctly excludes diq/ha/ku/uz/uz_AF
RTL_CODES = set(_lang_meta.loc[_lang_meta["directionality"] == "rtl", "language_code"])

# CJK / SE Asian: derive from primary_script
_CJK_SCRIPTS = {
    "Bopomofo", "Simplified", "Traditional", "Japanese", "Korean", "Katakana",
    "Tibetan", "Myanmar", "Khmer", "Lao", "Thai", "Han",
}
_cjk_from_csv = set(_lang_meta.loc[_lang_meta["primary_script"].isin(_CJK_SCRIPTS), "language_code"].dropna())
# Chinese variant codes present in pipeline data but missing primary_script in the metadata CSV
_CJK_EXTRA = {"zh-tw", "zh-classical", "zh-min-nan", "zh-yue", "cdo"}
CJK_CODES = _cjk_from_csv | _CJK_EXTRA

print(f"RTL codes: {len(RTL_CODES)}  |  CJK/SE Asian codes: {len(CJK_CODES)}")

def script_group(code):
    if code in RTL_CODES:
        return "RTL"
    if code in CJK_CODES:
        return "CJK / SE Asian"
    return "LTR (Latin/other)"

term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
svc_cols = {s: c for s, c in HEATMAP_SERVICES.items() if c in baseline.columns}
baseline["n_services"] = baseline[[c for c in svc_cols.values()]].notna().sum(axis=1)
baseline["script"] = baseline["language_code"].apply(script_group)

script_rows = []
for service, col in svc_cols.items():
    for script, grp in baseline.groupby("script"):
        n = int(grp[col].notna().sum())
        total = len(grp)
        script_rows.append({"service": service, "script": script, "n": n, "total": total})
script_df = pd.DataFrame(script_rows)

script_order = ["LTR (Latin/other)", "RTL", "CJK / SE Asian"]
svc_order = list(HEATMAP_SERVICES.keys())
max_n_script = script_df["n"].max()

heatmap = alt.Chart(script_df).mark_rect().encode(
    x=alt.X("service:N", sort=svc_order, title=None),
    y=alt.Y("script:N", sort=script_order, title=None),
    color=alt.Color("n:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, max_n_script]),
        title="languages translated"),
    tooltip=["service:N","script:N",
             alt.Tooltip("n:Q",title="translated"),
             alt.Tooltip("total:Q",title="total in group")],
)
hmap_text = heatmap.mark_text(fontSize=10).encode(
    text="n:Q",
    color=alt.condition(alt.datum.n > max_n_script * 0.6, alt.value("white"), alt.value("black")),
)

box = alt.Chart(baseline).mark_boxplot(extent="min-max").encode(
    x=alt.X("script:N", sort=script_order, title=None),
    y=alt.Y("n_services:Q", title="services that translated it", scale=alt.Scale(domain=[0,9])),
    color=alt.Color("script:N", sort=script_order, legend=None),
    tooltip=["script:N"],
).properties(width=260, height=220, title="Coverage depth by script group")

(heatmap + hmap_text).properties(
    width=480, height=100,
    title="Languages translated per service by script group",
) & box

## 1.9 Exact-Match Agreement Preview

For languages where 2+ services produced a translation, how often do they agree exactly? High agreement here means the pipeline is converging on clear answers for most languages.

In [11]:
term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
baseline["language_family"] = baseline["language_code"].apply(get_language_family)

all_svc_cols = {s: c for s, c in HEATMAP_SERVICES.items() if c in baseline.columns}

agreement_rows = []
for _, row in baseline.iterrows():
    vals = [row[c] for c in all_svc_cols.values() if pd.notna(row.get(c))]
    if len(vals) < 2: continue
    mode_val = max(set(vals), key=vals.count)
    n_agree = vals.count(mode_val)
    agreement_rows.append({
        "language_code": row["language_code"],
        "language_name": row.get("language_name", row["language_code"]),
        "language_family": row["language_family"],
        "n_services": len(vals),
        "n_agree": n_agree,
        "agreement_rate": round(n_agree / len(vals), 3),
        "best_candidate": mode_val,
    })
agree_df = pd.DataFrame(agreement_rows)

print(f"Languages with 2+ services: {len(agree_df)}")
print(f"Full agreement (all services match): {(agree_df['agreement_rate'] == 1.0).sum()}")
print(f"Majority agreement (≥60%): {(agree_df['agreement_rate'] >= 0.6).sum()}")
print(f"Low agreement (<60%): {(agree_df['agreement_rate'] < 0.6).sum()}")

# Distribution of agreement rates
hist = alt.Chart(agree_df).mark_bar().encode(
    x=alt.X("agreement_rate:Q", bin=alt.Bin(extent=[0,1], step=0.1), title="Agreement rate (fraction of services matching best candidate)"),
    y=alt.Y("count():Q", title="Number of languages"),
    color=alt.condition(
        alt.datum.agreement_rate >= 0.6, alt.value("#388e3c"), alt.value("#d32f2f")),
    tooltip=[alt.Tooltip("agreement_rate:Q",bin=alt.Bin(extent=[0,1],step=0.1)), "count():Q"],
).properties(width=380, height=220, title="Distribution of cross-service agreement rates")

# Mean agreement by language family
fam_agree = agree_df.groupby("language_family")["agreement_rate"].mean().reset_index().sort_values("agreement_rate")
fam_bar = alt.Chart(fam_agree).mark_bar().encode(
    y=alt.Y("language_family:N", sort=alt.EncodingSortField("agreement_rate", order="ascending"), title=None),
    x=alt.X("agreement_rate:Q", scale=alt.Scale(domain=[0,1]), title="Mean agreement rate"),
    color=alt.condition(
        alt.datum.agreement_rate >= 0.6, alt.value("#388e3c"), alt.value("#d32f2f")),
    tooltip=["language_family:N", alt.Tooltip("agreement_rate:Q", format=".2f")],
).properties(width=300, height=420, title="Mean agreement by language family")

hist | fam_bar

Languages with 2+ services: 858
Full agreement (all services match): 7
Majority agreement (≥60%): 89
Low agreement (<60%): 769


alt.HConcatChart(...)